# Change Data to Class

Sorts a YOLO-format detection dataset into per-class image folders by choosing, for each image, the class with the largest total bounding-box area.

In [ ]:
import os
import shutil
from collections import Counter

dataset_path = r"D:\Desktop\Roboflow - Sesame Dataset" 
output_path = r"D:\Desktop\Essa_Ahmed_Dominant_Class" 

classes = ['bacterial_leaf_blight', 'powdery_mildew']

os.makedirs(output_path, exist_ok=True)
for c in classes: 
    os.makedirs(os.path.join(output_path, c), exist_ok=True)

stats = Counter()

print("Starting Area-Weighted sorting...")

for folder in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset_path, folder, 'images')
    lbl_dir = os.path.join(dataset_path, folder, 'labels')
    
    if not os.path.exists(img_dir): continue

    for img_file in os.listdir(img_dir):
        if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        
        label_file = img_file.rsplit('.', 1)[0] + '.txt'
        label_path = os.path.join(lbl_dir, label_file)
        
        if not os.path.exists(label_path): continue

        class_areas = {i: 0.0 for i in range(len(classes))}

        with open(label_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.split()
                if len(parts) < 5: continue
                
                cls_idx = int(parts[0])
                width = float(parts[3])
                height = float(parts[4])
                area = width * height
                
                if cls_idx in class_areas:
                    class_areas[cls_idx] += area

        dominant_cls_idx = max(class_areas, key=class_areas.get)
        total_area_found = class_areas[dominant_cls_idx]

        if total_area_found > 0:
            class_name = classes[dominant_cls_idx]
            
            src_path = os.path.join(img_dir, img_file)
            dst_path = os.path.join(output_path, class_name, img_file)
            
            shutil.copy2(src_path, dst_path)
            stats[class_name] += 1

print("-" * 30)
print(f"Sorting Complete!")
print(f"Dominant images found:")
for cls_name, count in stats.items():
    print(f" - {cls_name}: {count} images")

## Infection-prioritised variant

Same area-weighted logic, but images with no (or empty) label file are routed to the *healthy* class.

In [ ]:
import os
import shutil
from collections import Counter

dataset_path = r"D:\Desktop\Roboflow (test-mrio3)" 
output_path = r"D:\Desktop\Roboflow (test-mrio3) Classified" 

classes = ['bacterial', 'healthy', 'powdery']

os.makedirs(output_path, exist_ok=True)
for c in classes: 
    os.makedirs(os.path.join(output_path, c), exist_ok=True)
os.makedirs(os.path.join(output_path, 'healthy'), exist_ok=True)

stats = Counter()

print("Starting Infection-Prioritized sorting...")

for folder in ['train', 'valid', 'test']:
    img_dir = os.path.join(dataset_path, folder, 'images')
    lbl_dir = os.path.join(dataset_path, folder, 'labels')
    
    if not os.path.exists(img_dir): continue

    for img_file in os.listdir(img_dir):
        if not img_file.lower().endswith(('.jpg', '.jpeg', '.png')): continue
        
        label_file = img_file.rsplit('.', 1)[0] + '.txt'
        label_path = os.path.join(lbl_dir, label_file)
        src_path = os.path.join(img_dir, img_file)

        if not os.path.exists(label_path) or os.path.getsize(label_path) == 0:
            dst_path = os.path.join(output_path, 'healthy', img_file)
            shutil.copy2(src_path, dst_path)
            stats['healthy'] += 1
            continue

        class_areas = {i: 0.0 for i in range(len(classes))}
        with open(label_path, 'r') as f:
            lines = f.readlines()
            for line in lines:
                parts = line.split()
                if len(parts) < 5: continue
                
                cls_idx = int(parts[0])
                width = float(parts[3])
                height = float(parts[4])
                class_areas[cls_idx] += (width * height)

        dominant_cls_idx = max(class_areas, key=class_areas.get)
        
        if class_areas[dominant_cls_idx] > 0:
            class_name = classes[dominant_cls_idx]
            dst_path = os.path.join(output_path, class_name, img_file)
            shutil.copy2(src_path, dst_path)
            stats[class_name] += 1
        else:
            dst_path = os.path.join(output_path, 'healthy', img_file)
            shutil.copy2(src_path, dst_path)
            stats['healthy'] += 1

print("-" * 30)
print(f"Sorting Complete!")
print(f"Final Class Counts for Apollo 14:")
for cls_name, count in stats.items():
    print(f" - {cls_name}: {count} images")